# Module 12 — Advanced Audio Classification

**Objectives:**
- Work with Mel Spectrogram image representations
- Generate and visualize spectrogram images
- Apply a deeper CNN to spectrogram images
- Perform data augmentation (noise, pitch shift, time stretch)
- Improve model generalisation
- Retrain the optimised model and compare with prior results

**Pipeline:** `Raw Audio → Mel Spectrogram → Data Augmentation → CNN → Evaluation`

In [ ]:
# ── Cell 1: Imports & Configuration ────────────────────────────────────────
import os, sys, time, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import librosa
import librosa.display
import joblib

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import layers, callbacks, regularizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)
from sklearn.preprocessing import LabelEncoder

# ── Paths ───────────────────────────────────────────────────────────────────
BASE_DIR    = os.path.abspath('..')
FEAT_DIR    = os.path.join(BASE_DIR, 'features')
MODEL_DIR   = os.path.join(BASE_DIR, 'models')
REP_DIR     = os.path.join(BASE_DIR, 'reports', 'advanced')
SPEC_DIR    = os.path.join(BASE_DIR, 'reports', 'spectrograms_adv')
os.makedirs(REP_DIR,  exist_ok=True)
os.makedirs(SPEC_DIR, exist_ok=True)

# ── Audio config ────────────────────────────────────────────────────────────
SR           = 22050          # sampling rate
DURATION     = 3.0            # fixed clip length (seconds)
N_SAMPLES    = int(SR * DURATION)
N_MELS       = 128            # mel frequency bins
N_FFT        = 2048
HOP_LENGTH   = 512
FMAX         = 8000           # max frequency (Hz)

# ── Training config ─────────────────────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.20
VAL_SIZE     = 0.15
BATCH_SIZE   = 32
EPOCHS       = 50

# ── Dark theme ──────────────────────────────────────────────────────────────
DARK   = '#0f172a'
PANEL  = '#1e293b'
ACCENT = '#818cf8'
plt.rcParams.update({'figure.facecolor': DARK, 'axes.facecolor': PANEL,
                     'text.color': 'white',     'axes.labelcolor': 'white',
                     'xtick.color': '#94a3b8',  'ytick.color': '#94a3b8',
                     'axes.edgecolor': '#334155','grid.color': '#334155',
                     'font.family': 'DejaVu Sans'})

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print('=' * 60)
print('  MODULE 12 — ADVANCED AUDIO CLASSIFICATION')
print('=' * 60)
print(f'  TensorFlow : {tf.__version__}')
print(f'  Librosa    : {librosa.__version__}')
print(f'  Sample Rate: {SR} Hz  |  Duration: {DURATION}s')
print(f'  Mel bins   : {N_MELS}   |  n_fft: {N_FFT}  |  hop: {HOP_LENGTH}')
print(f'  Output dir : {REP_DIR}')
print('=' * 60)

In [ ]:
# ── Cell 2: Load Dataset Metadata & Prepare File List ──────────────────────
meta = pd.read_csv(os.path.join(FEAT_DIR, 'dataset_metadata.csv'))

# Fix filepaths — replace notebooks/../data/ with correct absolute Data/Raw path
meta['filepath'] = meta['filepath'].apply(
    lambda p: p.replace(
        os.path.join(BASE_DIR, 'notebooks', '..', 'data', 'Raw'),
        os.path.join(BASE_DIR, 'Data', 'Raw')
    ).replace(
        os.path.join(BASE_DIR, 'notebooks\\..\\data\\Raw'),
        os.path.join(BASE_DIR, 'Data', 'Raw')
    )
)
# Normalise path separators
meta['filepath'] = meta['filepath'].apply(os.path.normpath)

# Verify files exist
meta['exists'] = meta['filepath'].apply(os.path.exists)
missing = meta[~meta['exists']]
if len(missing) > 0:
    print(f'WARNING: {len(missing)} files not found. Example: {missing.filepath.iloc[0]}')
else:
    print(f'All {len(meta)} audio files verified.')

meta = meta[meta['exists']].reset_index(drop=True)

EMOTIONS = sorted(meta['emotion'].unique())
N_CLASSES = len(EMOTIONS)
emotion_to_idx = {e: i for i, e in enumerate(EMOTIONS)}

# Load label encoder from Module 7
le = joblib.load(os.path.join(MODEL_DIR, 'label_encoder.pkl'))

print(f'Emotions ({N_CLASSES}): {EMOTIONS}')
print(f'Dataset size: {len(meta)} files')
print()
print(meta['emotion'].value_counts().to_string())

In [ ]:
# ── Cell 3: Generate & Visualise Mel Spectrogram Images ────────────────────
def load_audio_fixed(filepath, sr=SR, duration=DURATION):
    """Load audio, fix length, return waveform."""
    y, _ = librosa.load(filepath, sr=sr, mono=True, duration=duration)
    # Pad or trim to exactly N_SAMPLES
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)), mode='constant')
    else:
        y = y[:N_SAMPLES]
    return y

def compute_mel_spectrogram(y, sr=SR, n_mels=N_MELS, n_fft=N_FFT,
                             hop_length=HOP_LENGTH, fmax=FMAX):
    """Compute log-scaled Mel spectrogram (dB)."""
    S = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=n_mels, n_fft=n_fft,
        hop_length=hop_length, fmax=fmax
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    return S_db

# Sample one file per emotion and visualise
EMOT_COLORS = ['#818cf8','#34d399','#f87171','#fbbf24',
               '#a78bfa','#38bdf8','#fb923c','#f472b6']

sample_files = {}
for em in EMOTIONS:
    row = meta[meta['emotion'] == em].iloc[0]
    sample_files[em] = row['filepath']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.patch.set_facecolor(DARK)
fig.suptitle('Mel Spectrogram Representations — One Sample per Emotion',
             fontsize=14, fontweight='bold', color='white', y=1.01)

for ax, (em, fpath), color in zip(axes.flat, sample_files.items(), EMOT_COLORS):
    y_wav = load_audio_fixed(fpath)
    S_db  = compute_mel_spectrogram(y_wav)
    ax.set_facecolor(PANEL)
    img = librosa.display.specshow(
        S_db, sr=SR, hop_length=HOP_LENGTH, x_axis='time', y_axis='mel',
        fmax=FMAX, ax=ax, cmap='magma'
    )
    ax.set_title(em.capitalize(), fontsize=11, fontweight='bold',
                 color=color, pad=4)
    ax.set_xlabel('Time (s)', color='#94a3b8', fontsize=8)
    ax.set_ylabel('Hz', color='#94a3b8', fontsize=8)

plt.tight_layout()
out_path = os.path.join(REP_DIR, 'mel_spectrograms_per_emotion.png')
plt.savefig(out_path, dpi=130, bbox_inches='tight', facecolor=DARK)
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# ── Cell 4: Data Augmentation Functions ────────────────────────────────────
def add_white_noise(y, noise_factor=0.005):
    """Add Gaussian white noise to the waveform."""
    noise = np.random.randn(len(y))
    return y + noise_factor * noise

def add_colored_noise(y, noise_factor=0.003):
    """Add pink-ish noise (coloured) for more realistic augmentation."""
    noise = np.random.randn(len(y))
    # 1/f filter approximation
    fft_noise = np.fft.rfft(noise)
    freqs = np.fft.rfftfreq(len(noise))
    freqs[0] = 1e-6  # avoid division by zero
    power = 1.0 / np.sqrt(freqs)
    fft_noise *= power
    noise = np.fft.irfft(fft_noise, n=len(y))
    noise /= (np.max(np.abs(noise)) + 1e-8)
    return y + noise_factor * noise

def time_stretch(y, rate=None):
    """Time-stretch audio (slow down or speed up)."""
    if rate is None:
        rate = np.random.uniform(0.85, 1.15)
    y_stretched = librosa.effects.time_stretch(y, rate=rate)
    # Re-pad / trim to fixed length
    if len(y_stretched) < N_SAMPLES:
        y_stretched = np.pad(y_stretched, (0, N_SAMPLES - len(y_stretched)))
    else:
        y_stretched = y_stretched[:N_SAMPLES]
    return y_stretched

def pitch_shift(y, sr=SR, n_steps=None):
    """Shift pitch up or down by n semitones."""
    if n_steps is None:
        n_steps = np.random.uniform(-3, 3)
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def random_shift(y, shift_max=0.1):
    """Randomly shift the audio in time."""
    shift = int(np.random.uniform(-shift_max, shift_max) * len(y))
    return np.roll(y, shift)

def augment_waveform(y, aug_type=None):
    """Apply a FAST augmentation (noise/shift only) for bulk dataset building.
    time_stretch & pitch_shift are used only in the visual demo (Cell 4)."""
    fast_fns = [
        lambda y: add_white_noise(y, np.random.uniform(0.002, 0.008)),
        lambda y: add_colored_noise(y, np.random.uniform(0.002, 0.006)),
        lambda y: random_shift(y, shift_max=0.10),
        lambda y: add_white_noise(random_shift(y, 0.05), 0.003),
        lambda y: add_colored_noise(random_shift(y, 0.08), 0.003),
        lambda y: add_white_noise(y, np.random.uniform(0.001, 0.005)),
    ]
    if aug_type is not None:
        fn = fast_fns[aug_type % len(fast_fns)]
    else:
        fn = fast_fns[np.random.randint(len(fast_fns))]
    return fn(y)

# ── Visual comparison: original vs augmented ───────────────────────────────
sample_em = 'angry'
y_orig = load_audio_fixed(sample_files[sample_em])

aug_labels = ['Original', 'White Noise', 'Colored Noise', 'Time Stretch',
              'Pitch Shift', 'Random Shift']
aug_waves = [
    y_orig,
    add_white_noise(y_orig, 0.006),
    add_colored_noise(y_orig, 0.004),
    time_stretch(y_orig, 0.85),
    pitch_shift(y_orig, n_steps=2),
    random_shift(y_orig, 0.10),
]

fig, axes = plt.subplots(6, 2, figsize=(18, 18))
fig.patch.set_facecolor(DARK)
fig.suptitle(f'Data Augmentation — Waveform & Mel Spectrogram ({sample_em.capitalize()})',
             fontsize=14, fontweight='bold', color='white', y=1.01)

t = np.linspace(0, DURATION, N_SAMPLES)
row_colors = ['#94a3b8','#fbbf24','#fb923c','#34d399','#818cf8','#f472b6']

for row, (label, y_aug, col) in enumerate(zip(aug_labels, aug_waves, row_colors)):
    # Waveform
    ax_w = axes[row][0]
    ax_w.set_facecolor(PANEL)
    ax_w.plot(t, y_aug, color=col, linewidth=0.6, alpha=0.9)
    ax_w.set_title(label, fontsize=10, fontweight='bold', color=col, pad=3)
    ax_w.set_ylabel('Amplitude', color='#94a3b8', fontsize=8)
    ax_w.set_xlim(0, DURATION)
    if row == 5:
        ax_w.set_xlabel('Time (s)', color='#94a3b8', fontsize=8)

    # Mel Spectrogram
    ax_s = axes[row][1]
    ax_s.set_facecolor(PANEL)
    S_db = compute_mel_spectrogram(y_aug)
    librosa.display.specshow(
        S_db, sr=SR, hop_length=HOP_LENGTH, x_axis='time', y_axis='mel',
        fmax=FMAX, ax=ax_s, cmap='magma'
    )
    ax_s.set_title(f'{label} — Mel Spec', fontsize=10,
                   fontweight='bold', color=col, pad=3)
    if row == 5:
        ax_s.set_xlabel('Time (s)', color='#94a3b8', fontsize=8)

plt.tight_layout()
out_path = os.path.join(REP_DIR, 'augmentation_comparison.png')
plt.savefig(out_path, dpi=120, bbox_inches='tight', facecolor=DARK)
plt.show()
print(f'Saved: {out_path}')
print('\nAugmentation functions defined:')
for name in aug_labels[1:]:
    print(f'  [{name}]')

In [ ]:
# ── Cell 5: Build Mel Spectrogram Dataset (Original + Augmented) ───────────
def extract_mel_feature(filepath, augment=False, aug_type=None):
    """Load audio, optionally augment, return 2-D Mel spectrogram."""
    try:
        y = load_audio_fixed(filepath)
        if augment:
            y = augment_waveform(y, aug_type=aug_type)
        S_db = compute_mel_spectrogram(y)
        # Normalise to [0, 1]
        S_db = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-8)
        return S_db  # shape: (N_MELS, time_frames)
    except Exception as e:
        return None

print('Building Mel Spectrogram dataset...')
print(f'  Original files : {len(meta)}')

# Determine the number of time frames from one example
sample_spec = extract_mel_feature(meta['filepath'].iloc[0])
TIME_FRAMES  = sample_spec.shape[1]
INPUT_SHAPE  = (N_MELS, TIME_FRAMES, 1)   # H x W x C (channel-last)
print(f'  Spectrogram shape: {INPUT_SHAPE} (N_MELS x TIME_FRAMES x 1)')

# ── Extract original spectrograms ──────────────────────────────────────────
t0 = time.time()
X_orig, y_orig_labels = [], []

for _, row in meta.iterrows():
    spec = extract_mel_feature(row['filepath'], augment=False)
    if spec is not None:
        X_orig.append(spec)
        y_orig_labels.append(row['emotion'])

# ── Augment minority classes (neutral has only 96 samples vs 192 for others)
# We generate 2 augmented copies for neutral, 1 copy for all others
X_aug, y_aug_labels = [], []
emotion_counts = meta['emotion'].value_counts()

AUG_PER_EMOTION = {em: (2 if emotion_counts[em] < 150 else 1) for em in EMOTIONS}
print('  Augmentation plan:', AUG_PER_EMOTION)

for _, row in meta.iterrows():
    em = row['emotion']
    n_copies = AUG_PER_EMOTION[em]
    for k in range(n_copies):
        spec = extract_mel_feature(row['filepath'], augment=True, aug_type=k)
        if spec is not None:
            X_aug.append(spec)
            y_aug_labels.append(em)

elapsed = time.time() - t0

# ── Combine ────────────────────────────────────────────────────────────────
X_all = np.array(X_orig + X_aug, dtype=np.float32)[..., np.newaxis]  # add channel dim
y_all = np.array(y_orig_labels + y_aug_labels)

# Encode labels
y_all_enc = np.array([emotion_to_idx[e] for e in y_all], dtype=np.int32)

print(f'  Original samples  : {len(X_orig)}')
print(f'  Augmented samples : {len(X_aug)}')
print(f'  Total dataset     : {len(X_all)}')
print(f'  X shape           : {X_all.shape}  dtype={X_all.dtype}')
print(f'  Feature range     : [{X_all.min():.3f}, {X_all.max():.3f}]')
print(f'  Extraction time   : {elapsed:.1f}s')
print()
unique, counts = np.unique(y_all, return_counts=True)
print('  Class distribution after augmentation:')
for em, cnt in zip(unique, counts):
    print(f'    {em:<12}: {cnt}')

In [ ]:
# ── Cell 6: Train / Validation / Test Split ─────────────────────────────────
from sklearn.model_selection import train_test_split

# First split: 80% train+val, 20% test (stratified)
X_tv, X_test, y_tv, y_test = train_test_split(
    X_all, y_all_enc,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_all_enc
)

# Second split: 85% train, 15% val (of train+val)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_tv
)

# One-hot encode labels for categorical crossentropy
y_train_cat = tf.keras.utils.to_categorical(y_train, N_CLASSES)
y_val_cat   = tf.keras.utils.to_categorical(y_val,   N_CLASSES)
y_test_cat  = tf.keras.utils.to_categorical(y_test,  N_CLASSES)

print('Dataset splits:')
print(f'  Train : {X_train.shape}  labels: {y_train.shape}')
print(f'  Val   : {X_val.shape}  labels: {y_val.shape}')
print(f'  Test  : {X_test.shape}  labels: {y_test.shape}')

# Visualise class distribution in splits
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor(DARK)
fig.suptitle('Class Distribution Across Splits', fontsize=13,
             fontweight='bold', color='white')

split_data = [('Train', y_train), ('Validation', y_val), ('Test', y_test)]
bar_colors = ['#818cf8', '#34d399', '#f87171']

for ax, (split_name, y_split), col in zip(axes, split_data, bar_colors):
    ax.set_facecolor(PANEL)
    unique_s, counts_s = np.unique(y_split, return_counts=True)
    em_labels = [EMOTIONS[i] for i in unique_s]
    bars = ax.bar(em_labels, counts_s, color=col, alpha=0.85, edgecolor='#1e293b')
    for bar, cnt in zip(bars, counts_s):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(cnt), ha='center', va='bottom', color='white', fontsize=8)
    ax.set_title(f'{split_name} (n={len(y_split)})', fontsize=11,
                 fontweight='bold', color=col)
    ax.set_ylabel('Count', color='#94a3b8')
    ax.tick_params(axis='x', rotation=35)
    ax.set_ylim(0, max(counts_s) * 1.15)

plt.tight_layout()
plt.savefig(os.path.join(REP_DIR, 'split_distribution.png'), dpi=120,
            bbox_inches='tight', facecolor=DARK)
plt.show()

In [ ]:
# ── Cell 7: Build Deep CNN on Mel Spectrogram ──────────────────────────────
def build_mel_cnn(input_shape, n_classes, dropout_rate=0.35):
    """
    Deep CNN for Mel Spectrogram emotion classification.

    Architecture:
      Input(128 x T x 1)
       => Conv Block 1: Conv2D(32, 3x3) -> BN -> ReLU -> MaxPool(2x2) -> Drop(0.25)
       => Conv Block 2: Conv2D(64, 3x3) -> BN -> ReLU -> MaxPool(2x2) -> Drop(0.25)
       => Conv Block 3: Conv2D(128,3x3) -> BN -> ReLU -> MaxPool(2x2) -> Drop(0.30)
       => Conv Block 4: Conv2D(256,3x3) -> BN -> ReLU -> GlobalAvgPool -> Drop(0.35)
       => Dense(512)  -> BN -> ReLU -> Drop(0.40)
       => Dense(256)  -> BN -> ReLU -> Drop(0.30)
       => Dense(n_classes) -> Softmax
    """
    inp = keras.Input(shape=input_shape, name='mel_spec_input')

    # Block 1
    x = layers.Conv2D(32, (3, 3), padding='same', name='conv1')(inp)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.Activation('relu', name='relu1')(x)
    x = layers.MaxPooling2D((2, 2), name='pool1')(x)
    x = layers.Dropout(0.25, name='drop1')(x)

    # Block 2
    x = layers.Conv2D(64, (3, 3), padding='same', name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.Activation('relu', name='relu2')(x)
    x = layers.MaxPooling2D((2, 2), name='pool2')(x)
    x = layers.Dropout(0.25, name='drop2')(x)

    # Block 3
    x = layers.Conv2D(128, (3, 3), padding='same', name='conv3')(x)
    x = layers.BatchNormalization(name='bn3')(x)
    x = layers.Activation('relu', name='relu3')(x)
    x = layers.MaxPooling2D((2, 2), name='pool3')(x)
    x = layers.Dropout(0.30, name='drop3')(x)

    # Block 4 (deeper)
    x = layers.Conv2D(256, (3, 3), padding='same', name='conv4')(x)
    x = layers.BatchNormalization(name='bn4')(x)
    x = layers.Activation('relu', name='relu4')(x)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(dropout_rate, name='drop4')(x)

    # Dense head
    x = layers.Dense(512, kernel_regularizer=regularizers.l2(1e-4), name='dense1')(x)
    x = layers.BatchNormalization(name='bn5')(x)
    x = layers.Activation('relu', name='relu5')(x)
    x = layers.Dropout(0.40, name='drop5')(x)

    x = layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4), name='dense2')(x)
    x = layers.BatchNormalization(name='bn6')(x)
    x = layers.Activation('relu', name='relu6')(x)
    x = layers.Dropout(0.30, name='drop6')(x)

    out = layers.Dense(n_classes, activation='softmax', name='output')(x)

    model = keras.Model(inputs=inp, outputs=out, name='MelCNN')
    return model

mel_cnn = build_mel_cnn(INPUT_SHAPE, N_CLASSES)

mel_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

mel_cnn.summary()

# Param count
total_params = mel_cnn.count_params()
print(f'\nTotal parameters: {total_params:,}')
print(f'Input shape     : {INPUT_SHAPE}')
print(f'Output classes  : {N_CLASSES} -> {EMOTIONS}')

In [ ]:
# ── Cell 8: Train Mel-CNN with Callbacks ───────────────────────────────────
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

ckpt_path  = os.path.join(MODEL_DIR, 'mel_cnn_best.keras')
final_path = os.path.join(MODEL_DIR, 'mel_cnn_final.keras')

mel_cnn_callbacks = [
    callbacks.ModelCheckpoint(
        ckpt_path, monitor='val_accuracy',
        save_best_only=True, verbose=0, mode='max'
    ),
    callbacks.EarlyStopping(
        monitor='val_loss', patience=20,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=8, min_lr=1e-6, verbose=1
    ),
]

print('Training Mel-CNN with augmented data...')
print(f'  Epochs     : {EPOCHS}  (early stopping patience=20)')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  Train size : {len(X_train)}')
print(f'  Val size   : {len(X_val)}')
print()

t0 = time.time()

mel_cnn_history = mel_cnn.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=mel_cnn_callbacks,
    verbose=1
)

train_time = time.time() - t0

# Save final model
mel_cnn.save(final_path)

best_val_acc = max(mel_cnn_history.history['val_accuracy'])
best_ep      = mel_cnn_history.history['val_accuracy'].index(best_val_acc) + 1
total_ep     = len(mel_cnn_history.history['accuracy'])

print(f'\nTraining complete in {train_time:.1f}s ({total_ep} epochs run)')
print(f'Best val_accuracy: {best_val_acc*100:.2f}% at epoch {best_ep}')

In [ ]:
# ── Cell 9: Evaluate Mel-CNN & Plot Training Curves ────────────────────────
# Load best checkpoint
best_mel_cnn = keras.models.load_model(ckpt_path)

# ── Training curves ─────────────────────────────────────────────────────────
hist = mel_cnn_history.history
ep   = range(1, len(hist['accuracy']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
fig.patch.set_facecolor(DARK)
fig.suptitle('Mel-CNN Training Curves (with Data Augmentation)',
             fontsize=13, fontweight='bold', color='white')

for ax, metric, title, c_train, c_val in [
    (ax1, 'loss',     'Loss',     '#f87171', '#fbbf24'),
    (ax2, 'accuracy', 'Accuracy', '#818cf8', '#34d399'),
]:
    ax.set_facecolor(PANEL)
    ax.plot(ep, hist[metric],          color=c_train, lw=1.8, label=f'Train {title}')
    ax.plot(ep, hist[f'val_{metric}'], color=c_val,   lw=1.8, label=f'Val {title}', linestyle='--')
    ax.set_xlabel('Epoch', color='#94a3b8')
    ax.set_ylabel(title,   color='#94a3b8')
    ax.set_title(title, fontsize=11, fontweight='bold', color='white')
    ax.legend(facecolor=DARK, edgecolor='#334155', labelcolor='white')
    ax.grid(True, alpha=0.25)
    # Mark best val epoch
    if metric == 'accuracy':
        bv = max(hist['val_accuracy'])
        be = hist['val_accuracy'].index(bv)
        ax.axvline(be + 1, color='#f472b6', linestyle=':', lw=1.2,
                   label=f'Best Val ({bv*100:.1f}%)')
        ax.legend(facecolor=DARK, edgecolor='#334155', labelcolor='white')

plt.tight_layout()
plt.savefig(os.path.join(REP_DIR, 'mel_cnn_training_curves.png'), dpi=120,
            bbox_inches='tight', facecolor=DARK)
plt.show()

# ── Evaluation metrics ──────────────────────────────────────────────────────
test_loss, test_acc = best_mel_cnn.evaluate(X_test, y_test_cat, verbose=0)
y_pred_prob = best_mel_cnn.predict(X_test, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)

prec  = f1_score(y_test, y_pred, average='weighted',
                 zero_division=0, labels=np.arange(N_CLASSES))
rec   = f1_score(y_test, y_pred, average='macro',
                 zero_division=0, labels=np.arange(N_CLASSES))
f1    = f1_score(y_test, y_pred, average='weighted',
                 zero_division=0, labels=np.arange(N_CLASSES))

print('\nMel-CNN Test Results:')
print(f'  Test Accuracy : {test_acc*100:.2f}%')
print(f'  Test F1 (wtd) : {f1*100:.2f}%')
print(f'  Test Loss     : {test_loss:.4f}')
print()
print(classification_report(
    y_test, y_pred,
    target_names=EMOTIONS,
    zero_division=0
))

# ── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor(DARK)
fig.suptitle('Mel-CNN Confusion Matrix', fontsize=13,
             fontweight='bold', color='white')

for ax, data, title, fmt in [
    (axes[0], cm,      'Counts',      'd'),
    (axes[1], cm_norm, 'Normalised',  '.2f'),
]:
    ax.set_facecolor(PANEL)
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap='magma',
        xticklabels=EMOTIONS, yticklabels=EMOTIONS,
        ax=ax, cbar=True, linewidths=0.5, linecolor=PANEL
    )
    ax.set_title(title, fontsize=11, fontweight='bold', color='white')
    ax.set_xlabel('Predicted', color='#94a3b8')
    ax.set_ylabel('True',      color='#94a3b8')
    ax.tick_params(colors='#94a3b8')
    plt.setp(ax.get_xticklabels(), rotation=35, ha='right')
    plt.setp(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(REP_DIR, 'mel_cnn_confusion_matrix.png'), dpi=120,
            bbox_inches='tight', facecolor=DARK)
plt.show()

In [ ]:
# ── Cell 10: Compare All Models ────────────────────────────────────────────
import joblib
from sklearn.metrics import f1_score, accuracy_score

# Load Module 11 optimised ML models and their scalers
scaler   = joblib.load(os.path.join(MODEL_DIR, 'scaler.pkl'))
le_pkl   = joblib.load(os.path.join(MODEL_DIR, 'label_encoder.pkl'))
svm_opt  = joblib.load(os.path.join(MODEL_DIR, 'svm_optimized.pkl'))
lr_opt   = joblib.load(os.path.join(MODEL_DIR, 'lr_optimized.pkl'))
rf_opt   = joblib.load(os.path.join(MODEL_DIR, 'rf_optimized.pkl'))

# Load Module 11 optimised DL models
ann_best = keras.models.load_model(os.path.join(MODEL_DIR, 'ann_best.keras'))
cnn_best = keras.models.load_model(os.path.join(MODEL_DIR, 'cnn_best.keras'))

# Load features CSV (used by ML models)
feat_df = pd.read_csv(os.path.join(FEAT_DIR, 'features_dataset.csv'))
label_col = 'emotion'
feature_cols = [c for c in feat_df.columns if c not in [
    'emotion', 'emotion_code', 'label_encoded', 'filepath', 'filename',
    'actor', 'gender', 'intensity', 'statement', 'repetition',
    'modality', 'vocal_channel', 'duration_sec'
]]

X_feat = feat_df[feature_cols].values
y_feat_raw = feat_df[label_col].values if label_col in feat_df.columns else feat_df['label_encoded'].values

# Use the same label encoder that ML models were trained with
if y_feat_raw.dtype == object:
    y_feat = le_pkl.transform(y_feat_raw)
else:
    y_feat = y_feat_raw.astype(int)

X_feat_scaled = scaler.transform(X_feat)

# Test split for ML features (use same random state)
_, X_ml_test, _, y_ml_test = train_test_split(
    X_feat_scaled, y_feat, test_size=TEST_SIZE,
    random_state=RANDOM_STATE, stratify=y_feat
)

# For deep learning: ANN/CNN use MFCC-shaped features from Module 08
# We'll use the Mel-CNN test set directly for DL comparison

# ── Evaluate each model ─────────────────────────────────────────────────────
results = {}

# ML models
for name, clf in [('LR (opt)', lr_opt), ('RF (opt)', rf_opt), ('SVM (opt)', svm_opt)]:
    pred = clf.predict(X_ml_test)
    results[name] = {
        'accuracy': accuracy_score(y_ml_test, pred) * 100,
        'f1':       f1_score(y_ml_test, pred, average='weighted', zero_division=0) * 100,
        'type':     'ML'
    }

# DL models (ANN / MFCC-CNN) on Mel-CNN test data
# Note: these models take flattened features — use ML test set shape via Mel spec
# For ANN: it was trained on flattened MFCC features (same scaler)
# Mel-CNN is trained on Mel spectrogram — use its own test split

# ANN uses flattened scaled features
try:
    ann_pred_prob = ann_best.predict(X_ml_test, verbose=0)
    ann_pred      = np.argmax(ann_pred_prob, axis=1)
    le_classes    = list(le_pkl.classes_)
    results['ANN (opt)'] = {
        'accuracy': accuracy_score(y_ml_test, ann_pred) * 100,
        'f1':       f1_score(y_ml_test, ann_pred, average='weighted', zero_division=0) * 100,
        'type':     'DL'
    }
except Exception as ann_err:
    print(f'ANN eval skipped: {ann_err}')
    results['ANN (opt)'] = {'accuracy': 0.0, 'f1': 0.0, 'type': 'DL'}

# Mel-CNN (Module 12) on its own test split
mel_pred      = np.argmax(best_mel_cnn.predict(X_test, verbose=0), axis=1)
results['Mel-CNN (M12)'] = {
    'accuracy': accuracy_score(y_test, mel_pred) * 100,
    'f1':       f1_score(y_test, mel_pred, average='weighted', zero_division=0) * 100,
    'type':     'DL (Mel+Aug)'
}

# ── Print comparison table ──────────────────────────────────────────────────
print('\n' + '=' * 55)
print(f'  {"Model":<18}  {"Type":<14}  {"Acc":>7}  {"F1":>7}')
print('  ' + '-' * 53)
for name, r in results.items():
    print(f'  {name:<18}  {r["type"]:<14}  {r["accuracy"]:>6.2f}%  {r["f1"]:>6.2f}%')
print('=' * 55)

# ── Bar chart comparison ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.patch.set_facecolor(DARK)
fig.suptitle('All Models Comparison — Module 12 vs Prior Modules',
             fontsize=13, fontweight='bold', color='white')

model_names = list(results.keys())
accs = [results[n]['accuracy'] for n in model_names]
f1s  = [results[n]['f1']       for n in model_names]

palette = ['#818cf8','#34d399','#f87171','#fbbf24','#f472b6']
x = np.arange(len(model_names))

for ax, vals, metric in [(axes[0], accs, 'Accuracy (%)'), (axes[1], f1s, 'F1-Score (weighted, %')]:
    ax.set_facecolor(PANEL)
    bars = ax.bar(x, vals, color=palette[:len(x)], alpha=0.9,
                  edgecolor='#1e293b', linewidth=0.8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{v:.1f}%', ha='center', va='bottom', color='white',
                fontsize=9, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(metric, color='#94a3b8')
    ax.set_title(metric, fontsize=11, fontweight='bold', color='white')
    ax.set_ylim(0, max(vals) * 1.15)
    ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(REP_DIR, 'all_models_comparison.png'), dpi=120,
            bbox_inches='tight', facecolor=DARK)
plt.show()

# Save comparison CSV
comp_df = pd.DataFrame([
    {'model': n, 'type': r['type'], 'accuracy': round(r['accuracy'], 2),
     'f1_weighted': round(r['f1'], 2)}
    for n, r in results.items()
])
comp_df.to_csv(os.path.join(REP_DIR, 'model_comparison.csv'), index=False)
print(f'\nComparison CSV saved: {os.path.join(REP_DIR, "model_comparison.csv")}')

In [ ]:
# ── Cell 11: Feature Map Visualisation ─────────────────────────────────────
# Visualise what the first conv layer has learned

# Build an intermediate model to extract first conv block activations
feature_extractor = keras.Model(
    inputs=best_mel_cnn.input,
    outputs=best_mel_cnn.get_layer('conv1').output
)

# Pick one test sample per emotion
n_show = min(4, N_CLASSES)
sample_indices = [np.where(y_test == i)[0][0] for i in range(n_show)]

fig = plt.figure(figsize=(20, n_show * 4))
fig.patch.set_facecolor(DARK)
fig.suptitle('Conv Layer 1 — Feature Map Activations (first 8 filters)',
             fontsize=13, fontweight='bold', color='white', y=1.01)

N_FILTERS_SHOW = 8
for row_idx, sample_idx in enumerate(sample_indices):
    sample_input = X_test[sample_idx:sample_idx+1]  # (1, H, W, 1)
    activations  = feature_extractor(sample_input, training=False)  # (1, H, W, 32)
    activations  = activations.numpy()[0]  # (H, W, 32)
    em_label     = EMOTIONS[y_test[sample_idx]]

    for col_idx in range(N_FILTERS_SHOW):
        ax = fig.add_subplot(n_show, N_FILTERS_SHOW + 1,
                             row_idx * (N_FILTERS_SHOW + 1) + col_idx + 2)
        ax.set_facecolor(PANEL)
        ax.imshow(activations[:, :, col_idx].T, aspect='auto',
                  cmap='magma', origin='lower')
        ax.set_xticks([])
        ax.set_yticks([])
        if col_idx == 0:
            ax.set_ylabel(em_label.capitalize(), color=EMOT_COLORS[y_test[sample_idx]],
                          fontsize=9, fontweight='bold')
        if row_idx == 0:
            ax.set_title(f'F{col_idx+1}', fontsize=9, color='#94a3b8')

    # Show original spectrogram
    ax0 = fig.add_subplot(n_show, N_FILTERS_SHOW + 1,
                          row_idx * (N_FILTERS_SHOW + 1) + 1)
    ax0.set_facecolor(PANEL)
    ax0.imshow(X_test[sample_idx, :, :, 0].T, aspect='auto',
               cmap='magma', origin='lower')
    ax0.set_title('Input\nMel Spec', fontsize=8, color='white')
    ax0.set_xticks([])
    ax0.set_yticks([])

plt.tight_layout()
plt.savefig(os.path.join(SPEC_DIR, 'feature_maps_conv1.png'), dpi=110,
            bbox_inches='tight', facecolor=DARK)
plt.show()
print(f'Feature map visualisation saved.')

In [ ]:
# ── Cell 12: Save Mel-CNN & Module Summary ─────────────────────────────────
import joblib

# Save best mel-cnn (already saved during training; re-confirm)
best_mel_cnn.save(os.path.join(MODEL_DIR, 'mel_cnn_best.keras'))

# Save the label encoder index mapping for Module 12 (mel cnn uses emotion_to_idx)
joblib.dump(emotion_to_idx, os.path.join(MODEL_DIR, 'mel_cnn_label_map.pkl'))

# Save augmentation config
aug_config = {
    'sr': SR, 'duration': DURATION, 'n_mels': N_MELS,
    'n_fft': N_FFT, 'hop_length': HOP_LENGTH, 'fmax': FMAX,
    'input_shape': list(INPUT_SHAPE),
    'emotions': EMOTIONS, 'n_classes': N_CLASSES,
    'emotion_to_idx': emotion_to_idx
}
with open(os.path.join(MODEL_DIR, 'mel_cnn_config.json'), 'w') as f:
    json.dump(aug_config, f, indent=2)

# ── List saved files ────────────────────────────────────────────────────────
model_files  = sorted(os.listdir(MODEL_DIR))
report_files = sorted(os.listdir(REP_DIR))

print('=' * 66)
print('   MODULE 12 - ADVANCED AUDIO CLASSIFICATION COMPLETE')
print('=' * 66)
print()
print('  Mel Spectrogram CNN (Module 12):')
best_mel_acc = max(mel_cnn_history.history['val_accuracy']) * 100
print(f'    Best Val Accuracy : {best_mel_acc:.2f}%')
print(f'    Test Accuracy     : {test_acc*100:.2f}%')
print(f'    Test F1 (wtd)     : {f1*100:.2f}%')
print(f'    Training Time     : {train_time:.1f}s')
print(f'    Model Parameters  : {total_params:,}')
print()
print('  Data Augmentation Applied:')
print('    + White Noise       (random sigma: 0.002-0.008)')
print('    + Colored (Pink) Noise')
print('    + Time Stretch      (rate: 0.85-1.15)')
print('    + Pitch Shift       (semitones: -3 to +3)')
print('    + Random Time Shift (10%)')
print(f'    Total samples after aug: {len(X_all)}')
print()
print('  All Models Comparison (Test F1, weighted):')
for name, r in results.items():
    marker = ' <-- BEST' if r['f1'] == max(res['f1'] for res in results.values()) else ''
    print(f'    {name:<18}: {r["f1"]:>6.2f}%{marker}')
print()
print('  Saved models:')
for f in model_files:
    sz = os.path.getsize(os.path.join(MODEL_DIR, f)) // 1024
    print(f'    {f:<45} {sz:>6} KB')
print()
print('  Saved reports (reports/advanced/):')
for f in report_files:
    sz = os.path.getsize(os.path.join(REP_DIR, f)) // 1024
    print(f'    {f:<45} {sz:>6} KB')
print()
print('  Module 12 Complete - Ready for Module 13 (Streamlit App)!')
print('=' * 66)